# Detecção de EPI — Demonstração

## 1. Instalar

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import cv2, numpy as np
from google.colab import files
print('pronto')

## 2. Enviar o modelo treinado (epi_yolo.pt)

In [ ]:
enviados = files.upload()
nome_modelo = next(iter(enviados))
modelo = YOLO(nome_modelo)
print('Modelo carregado. Classes:', modelo.names)

## 3. Lógica de conformidade

In [ ]:
def classificar(nome):
    n = nome.lower()
    if n.startswith("no"):                                   # NO-Hardhat / NO-Safety Vest -> ausencia
        return None
    if "person" in n or "pessoa" in n:                       return "pessoa"
    if "hardhat" in n or "helmet" in n or "capacete" in n:   return "capacete"
    if "vest" in n or "colete" in n:                         return "colete"
    return None

def centro_dentro(b, p):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return p[0] <= cx <= p[2] and p[1] <= cy <= p[3]

def overlay_deteccao(frame, conf=0.35):
    # Desenha caixas/status num overlay RGBA transparente (para sobrepor ao video ao vivo).
    over = np.zeros((frame.shape[0], frame.shape[1], 4), dtype=np.uint8)
    r = modelo(frame, conf=conf, verbose=False)[0]
    pes, cap, col = [], [], []
    if r.boxes is not None:
        for box, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)):
            t = classificar(modelo.names.get(int(c), str(c)))
            if t == "pessoa":   pes.append(box)
            elif t == "capacete": cap.append(box)
            elif t == "colete":   col.append(box)
    algum_nc = False
    for p in pes:
        tem_cap = any(centro_dentro(b, p) for b in cap)
        tem_col = any(centro_dentro(b, p) for b in col)
        conforme = tem_cap and tem_col
        algum_nc = algum_nc or (not conforme)
        cor = (0,200,0,255) if conforme else (255,0,0,255)   # RGBA
        x1,y1,x2,y2 = [int(v) for v in p]
        cv2.rectangle(over,(x1,y1),(x2,y2),cor,2)
        txt = ("capacete OK" if tem_cap else "sem capacete")+" | "+("colete OK" if tem_col else "sem colete")
        cv2.putText(over,txt,(x1,max(20,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,0.6,cor,2)
    status = "NAO CONFORME" if (algum_nc or not pes) else "CONFORME"
    cor_s = (255,0,0,255) if status=="NAO CONFORME" else (0,200,0,255)
    cv2.rectangle(over,(0,0),(over.shape[1],32),cor_s,-1)
    cv2.putText(over,"STATUS: "+status,(10,23),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255,255),2)
    return over
print("funcoes prontas")

## 4. Webcam ao vivo

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from PIL import Image
import io

def js_to_image(js_reply):
    b = b64decode(js_reply.split(",")[1])
    arr = np.frombuffer(b, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)

def bbox_to_bytes(over):
    im = Image.fromarray(over, "RGBA")
    buf = io.BytesIO(); im.save(buf, format="png")
    return "data:image/png;base64,{}".format(str(b64encode(buf.getvalue()), "utf-8"))

JS = '''
var video; var div = null; var stream; var captureCanvas; var imgElement; var labelElement;
var pendingResolve = null; var shutdown = false;
function removeDom(){ stream.getVideoTracks()[0].stop(); video.remove(); div.remove();
    video=null; div=null; stream=null; imgElement=null; captureCanvas=null; labelElement=null; }
function onAnimationFrame(){
    if(!shutdown){ window.requestAnimationFrame(onAnimationFrame); }
    if(pendingResolve){ var result="";
        if(!shutdown){ captureCanvas.getContext("2d").drawImage(video,0,0,640,480);
            result = captureCanvas.toDataURL("image/jpeg", 0.8); }
        var lp = pendingResolve; pendingResolve=null; lp(result); } }
async function createDom(){
    if(div!==null){ return stream; }
    div = document.createElement("div"); div.style.border="2px solid black";
    div.style.padding="3px"; div.style.width="100%"; div.style.maxWidth="600px";
    document.body.appendChild(div);
    const modelOut = document.createElement("div"); modelOut.innerHTML="<span>Status:</span>";
    labelElement = document.createElement("span"); labelElement.innerText="No data";
    labelElement.style.fontWeight="bold"; modelOut.appendChild(labelElement); div.appendChild(modelOut);
    video = document.createElement("video"); video.style.display="block";
    video.width = div.clientWidth-6; video.setAttribute("playsinline","");
    video.onclick = () => { shutdown = true; };
    stream = await navigator.mediaDevices.getUserMedia({video:{facingMode:"user"}});
    div.appendChild(video);
    imgElement = document.createElement("img"); imgElement.style.position="absolute";
    imgElement.style.zIndex=1; imgElement.onclick = () => { shutdown = true; }; div.appendChild(imgElement);
    const instruction = document.createElement("div");
    instruction.style.color="red"; instruction.style.fontWeight="bold";
    instruction.innerText="Clique no video para encerrar";
    div.appendChild(instruction); instruction.onclick = () => { shutdown = true; };
    video.srcObject = stream; await video.play();
    captureCanvas = document.createElement("canvas"); captureCanvas.width=640; captureCanvas.height=480;
    window.requestAnimationFrame(onAnimationFrame); return stream; }
async function stream_frame(label, imgData){
    if(shutdown){ removeDom(); shutdown=false; return ""; }
    stream = await createDom();
    if(label != ""){ labelElement.innerHTML = label; }
    if(imgData != ""){ var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px"; imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px"; imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData; }
    var result = await new Promise(function(resolve, reject){ pendingResolve = resolve; });
    shutdown = false; return {"img": result}; }
'''

def video_stream():
    display(Javascript(JS))

def video_frame(label, bbox):
    return eval_js("stream_frame('{}', '{}')".format(label, bbox))

# Loop AO VIVO: captura -> detecta -> sobrepoe as caixas no video. Clique no video para parar.
video_stream()
bbox = ""
while True:
    js_reply = video_frame("Detectando EPI...", bbox)
    if not js_reply:
        break
    frame = js_to_image(js_reply["img"])
    overlay = overlay_deteccao(frame)
    bbox = bbox_to_bytes(overlay)